
# 03 — Real Clinical Data, an Honest Surprise, and "The Optimizer's Curse"

**Prerequisite:** Notebooks `00`–`02`.

Everything so far was a synthetic physics system or an academic benchmark
suite. This notebook covers the project's test on **genuine clinical
diagnostic data**: Wisconsin Breast Cancer (569 patients) and Pima Indians
Diabetes (768 patients) — real binary classification tasks where a missed
positive case (a missed cancer, a missed diabetes diagnosis) is far more
costly than a false alarm.

This is also where the project found its most interesting, counter-intuitive
result — and rather than hide it, dug in and explained it. That explanation
is the heart of this notebook.


In [ ]:

import sys, os, json
import numpy as np

REPO_PATH = None
_candidates = [REPO_PATH, "HPO-HMC", os.path.join("..", "HPO-HMC"), "."]
for _c in _candidates:
    if _c and os.path.isdir(os.path.join(_c, "results")):
        REPO_PATH = _c
        break
if REPO_PATH is None:
    raise FileNotFoundError("Couldn't find the HPO-HMC repo. Set REPO_PATH manually.")
print(f"Using repo at: {os.path.abspath(REPO_PATH)}")

def load_json(*parts):
    with open(os.path.join(REPO_PATH, *parts)) as f:
        return json.load(f)

breast_cancer = load_json("results", "breast_cancer", "results.json")
diabetes = load_json("results", "diabetes", "results.json")
methods = ["Default Adam", "Random Search", "Optuna TPE", "Method C (HHD-ABBO)"]



## 1. The results, straight from the raw JSON (no numbers re-typed by hand)


In [ ]:

def summarize(dataset, name):
    print(f"=== {name} ===")
    print(f"{'Method':22s} {'Test AUROC':>14s} {'Positive Recall':>16s} {'Time (s)':>10s}")
    for m in methods:
        rows = [r for r in dataset if r["method"] == m and r.get("test_metrics")]
        auroc = np.array([r["test_metrics"]["auroc"] for r in rows])
        recall = np.array([r["test_metrics"]["positive_recall"] for r in rows])
        t = np.array([r["time"] for r in rows])
        print(f"{m:22s} {auroc.mean():>7.4f}+/-{auroc.std():<6.4f} "
              f"{recall.mean():>8.4f}+/-{recall.std():<6.4f} {t.mean():>9.1f}")
    print()

summarize(breast_cancer, "Wisconsin Breast Cancer (easy task, AUROC ceiling ~0.99)")
summarize(diabetes, "Pima Indians Diabetes (harder task, more headroom)")



### Look closely at the Diabetes table

**Default Adam — with zero hyperparameter search — has the highest mean
AUROC of all four methods.** A 20-trial Optuna search, and even Method C's
physics-based exploration, both came in slightly *behind* an untuned
baseline. That looks backwards. More search should mean better results, not
worse.

We checked this wasn't a data or computation error by re-deriving every
number directly from the raw per-seed JSON above — it isn't. So what's
actually going on? Let's build the explanation from first principles.



## 2. The optimizer's curse — built from scratch with fake data first

Here's the core intuition, with a deliberately simple simulation before we
touch the real data at all.

Imagine 20 hyperparameter configurations that are all, in truth, **exactly
equally good** — say, each has a true AUROC of 0.80. But we only get to
*measure* each one on a small, noisy validation set, so each measurement has
some random noise added. If we then pick whichever configuration *measured*
the highest, what do we expect that configuration's *true* performance to
actually be?


In [ ]:

rng = np.random.default_rng(0)

true_quality = 0.80          # every config is secretly, truly, equally good
noise_std = 0.03             # noise from a small, finite validation set

def simulate_trial(n_configs, n_repeats=20000):
    # Try n_configs configs (all secretly equal quality), measure each with
    # noise, pick the best-MEASURED one, and see how it does on a fresh
    # (noisy, but independent) 'test' draw.
    gaps = []
    for _ in range(n_repeats):
        val_scores  = true_quality + rng.normal(0, noise_std, size=n_configs)
        best_idx = np.argmax(val_scores)
        # the SAME config's quality on an independent ("test") draw of noise:
        test_score_of_chosen = true_quality + rng.normal(0, noise_std)
        gaps.append(val_scores[best_idx] - test_score_of_chosen)
    return np.mean(gaps)

for n in [1, 5, 20, 50]:
    gap = simulate_trial(n)
    print(f"Configs compared: {n:3d}  ->  avg (val - test) gap for the CHOSEN config: {gap:+.4f}")



Even though **every configuration is secretly, truly, identical**, the gap
between "how good it looked on validation" and "how good it actually is"
**grows with the number of configurations you compared** — because picking
the max of many noisy measurements systematically favors whichever one got
lucky. With only 1 "configuration" (no search at all), there's no such bias.
This is a real, well-known statistical phenomenon called **the optimizer's
curse** (or the winner's curse of model selection).

Now let's check whether this exact mechanism explains what happened on the
real Diabetes dataset.



## 3. Checking the real data for exactly this signature

Diabetes has only 768 patients total; the validation split used to pick the
best hyperparameters is roughly 154 patients — a small, genuinely noisy
scoreboard, just like the toy simulation above. If the optimizer's curse is
really what's happening, methods that tried *more* configurations against
that small validation set should show a *bigger* gap between their
validation score and their (independent) test score.


In [ ]:

def val_test_gap(dataset, name):
    print(f"=== {name}: validation-AUROC minus test-AUROC gap ===")
    print("(positive = validation looked better than test really was = overfit to validation)\n")
    for m in methods:
        rows = [r for r in dataset if r["method"] == m and r.get("test_metrics")]
        gaps = [r["val_auroc"] - r["test_metrics"]["auroc"] for r in rows]
        print(f"  {m:22s} mean gap = {np.mean(gaps):+.4f}  (std {np.std(gaps):.4f})")
    print()

val_test_gap(diabetes, "Diabetes")
val_test_gap(breast_cancer, "Breast Cancer")



### This is a remarkably clean confirmation

On Diabetes:
- **Default Adam** (0 configurations compared): gap ~ **0** (unbiased — exactly
  like "n=1" in the toy simulation above)
- **Method C** (continuous trajectory exploration, not discrete trials): a
  **small** positive gap
- **Optuna TPE** and **Random Search** (20 discrete trial-and-error configs
  each): **much larger** positive gaps — Random Search's is the worst of all

The ranking of "how much each method overfits its own validation set" lines
up almost exactly with "how many discrete configurations it compared against
a small noisy scoreboard" — which is exactly the optimizer's curse mechanism
demonstrated in the simulation above. On Breast Cancer, all the gaps are tiny
across every method, because that task is near the 0.99 AUROC ceiling — there's
much less noise to overfit to in the first place, which is why this effect
shows up clearly on the harder dataset and barely at all on the easier one.

**Why this is genuinely good news for Method C, hiding inside what looked
like bad news:** Method C explores hyperparameters through one continuous
physics-driven trajectory, not by repeatedly re-rolling the dice on
independent configurations — and its validation-overfitting bias is only a
third the size of the discrete-search methods. That's a real, structural
advantage of the approach, even in the one result where its raw accuracy
didn't come out on top.



## 4. But is any of this actually statistically significant?

With only 5 seeds, we should check rather than assume. A Friedman test
(exactly like Notebook `02`) on all four real-world comparisons came back:

| Comparison | Friedman p-value | Significant at 0.05? |
|---|---|---|
| Breast Cancer, AUROC | 0.472 | No |
| Breast Cancer, Recall | 0.212 | No |
| Diabetes, AUROC | 0.472 | No |
| Diabetes, Recall | 0.102 | No |

**None of the accuracy or recall differences among the four methods are
statistically significant on either dataset.** This must be read in both
directions: this does *not* prove Method C is better, and it does *not*
prove it's worse. What *is* real and consistent (not a statistical test, but
a direct wall-clock measurement): Method C finishes in ~2 seconds vs.
~22–29 seconds for the 20-trial searches — a genuine 13–19x speed advantage
that holds on both datasets.

**The honest, defensible headline: "Method C matches 20-trial search quality
at a fraction of the cost" — not "Method C is more accurate."**

**Next notebook (`04`):** the project also tried upgrading its sampler to
NUTS (a fancier, adaptive alternative to the fixed-step leapfrog from
Notebook `00`) — and it lost. Let's see why, live.
